<a href="https://colab.research.google.com/github/Lobnaait/SEARCH_Lobna_Tsetline_CMRI/blob/main/Tsetline_ACDCdataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from huggingface_hub import snapshot_download
data_dir = snapshot_download(repo_id="mathpluscode/ACDC", allow_patterns=["*.nii.gz", "*.csv"], repo_type="dataset")
# https://huggingface.co/datasets/mathpluscode/ACDC

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 752 files:   0%|          | 0/752 [00:00<?, ?it/s]

In [3]:
!ls /root/.cache/huggingface/hub/datasets--mathpluscode--ACDC/snapshots/4a312b610f395eda5dc0aa36f087dd6ca9991e41/

test  test.csv	train  train.csv


In [4]:
!pip install pyTsetlinMachine

  Preparing metadata (setup.py) ... done
  Created wheel for pyTsetlinMachine: filename=pytsetlinmachine-0.6.6-cp313-cp313-linux_x86_64.whl size=59784 sha256=47e4dbed842235d62dc348d22ee754b1b5b25970c78c57ec5b03e9cc177c4ab1
  Stored in directory: /root/.cache/pip/wheels/a9/66/98/535c2cc844fdb6fc12f31feefbaa6a33222b1378914098f498
Successfully built pyTsetlinMachine


In [5]:
#IMPORT LIBRARIES AND DATASET
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, auc, roc_auc_score, classification_report, confusion_matrix
from pyTsetlinMachine.tm import MultiClassTsetlinMachine

In [6]:
import os

train = pd.read_csv(os.path.join(data_dir, "train.csv"))
test = pd.read_csv(os.path.join(data_dir, "test.csv"))

print(train.shape)
print(test.shape)

train.head(10)

(100, 15)
(50, 15)


,pid,pathology,height,weight,bmi,n_frames,ed_frame,es_frame,original_sax_spacing_x,original_sax_spacing_y,original_sax_spacing_z,n_slices,edv,esv,ef
0,patient001,DCM,1.84,95.0,28.060019,30,1,12,1.562500,1.562500,10.0,10,1354.58,1125.97,16.876818
1,patient002,DCM,1.60,70.0,27.343750,30,1,12,1.367188,1.367188,10.0,10,1212.69,979.53,19.226678
2,patient003,DCM,1.65,77.0,28.282828,30,1,15,1.562500,1.562500,10.0,10,1405.42,1298.96,7.574960
3,patient004,DCM,1.59,46.0,18.195483,28,1,15,1.367188,1.367188,10.0,10,1226.58,1114.83,9.110698
4,patient005,DCM,1.65,77.0,28.282828,30,1,13,1.406250,1.406250,10.0,10,1446.55,1211.36,16.258684
5,patient006,DCM,1.80,70.0,21.604938,28,1,16,1.757812,1.757812,10.0,11,1637.74,1374.18,16.092909
6,patient007,DCM,1.73,107.0,35.751278,16,1,7,1.875000,1.875000,10.0,10,1589.67,1411.75,11.192260
7,patient008,DCM,1.80,100.0,30.864198,28,1,13,1.562500,1.562500,10.0,10,1332.23,1158.90,13.010516
8,patient009,DCM,1.53,61.0,26.058354,35,1,13,1.367190,1.367190,10.0,10,1254.25,1154.37,7.963325
9,patient010,DCM,1.70,68.0,23.529412,28,1,13,1.562500,1.562500,10.0,10,1484.35,1316.59,11.301917


In [7]:
TARGET = "pathology"
print(train[TARGET].value_counts())


pathology
DCM     20
HCM     20
MINF    20
NOR     20
RV      20
Name: count, dtype: int64


In [10]:
# Convert diagnosis labels to numbers
#label_encoder = LabelEncoder()

#y = label_encoder.fit_transform(y)

#print("\nClass mapping:")
#for i, name in enumerate(label_encoder.classes_):
 #   print(i, "->", name)

In [11]:
print(train.columns.tolist())
features = train.columns.tolist()
features.remove(TARGET)
features.remove('pid')
print(features)

print("\n ")
X = train[features].copy()
y = train[TARGET].copy()

print("X_train shape:", X.shape)
print("y_train shape:", y.shape)
print("\nClasses:", y.value_counts())

X_train, X_validation, y_train, y_validation = train_test_split(X, y, test_size=0.25, stratify=y)

['pid', 'pathology', 'height', 'weight', 'bmi', 'n_frames', 'ed_frame', 'es_frame', 'original_sax_spacing_x', 'original_sax_spacing_y', 'original_sax_spacing_z', 'n_slices', 'edv', 'esv', 'ef']
['height', 'weight', 'bmi', 'n_frames', 'ed_frame', 'es_frame', 'original_sax_spacing_x', 'original_sax_spacing_y', 'original_sax_spacing_z', 'n_slices', 'edv', 'esv', 'ef']

 
X_train shape: (100, 13)
y_train shape: (100,)

Classes: pathology
DCM     20
HCM     20
MINF    20
NOR     20
RV      20
Name: count, dtype: int64


In [12]:
features = test.columns.tolist()
features.remove(TARGET)
features.remove('pid')

X_test = test[features].copy()
y_test = test[TARGET].copy()

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
print("\nClasses:", y_test.value_counts())


X_test shape: (50, 13)
y_test shape: (50,)

Classes: pathology
DCM     10
NOR     10
MINF    10
HCM     10
RV      10
Name: count, dtype: int64


In [41]:
def compute_thresholds(X, n_thresholds):
  percentile_values = np.linspace(0, 100, int(n_thresholds) + 2)[1:-1] # Exclude 0 and 100
  thresholds = np.percentile(X, percentile_values)
  return np.unique(thresholds)

In [14]:
# ---------------------------------------------------------
# Convert continuous features into binary features
# ---------------------------------------------------------
import numpy as np

def threshold_binarize(X, thresholds):
    binary_features = []

    for feature_index in range(X.shape[1]):
        for threshold in thresholds[feature_index]:
            binary_features.append( (X[:, feature_index] >= threshold).astype(np.uint32) )

    return np.array(binary_features).T

In [15]:
# ---------------------------------------------------------
# Find optimal number of thresholds
# ---------------------------------------------------------

candidate_thresholds = range(1, 11)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for n_thresholds in candidate_thresholds:
    fold_accuracies = []

    for train_index, val_index in cv.split(X, y):

        X_train = X[train_index]
        X_validation = X[val_index]

        y_train = y[train_index]
        y_validation = y[val_index]

        thresholds = create_thresholds(X_train,n_thresholds)

        # Binarize training and validation data
        X_train_bin = binarize( X_train,thresholds)
        X_val_bin = binarize(X_val,thresholds)

        # Create Tsetlin Machine
        tm = MultiClassTsetlinMachine(number_of_clauses=1000, T=50, s=5.0)

        # Train
        tm.fit(X_train_bin, y_train, epochs=100)

        # Validation prediction
        prediction = tm.predict(X_val_bin)

        accuracy = accuracy_score(y_validation, prediction)

        fold_accuracies.append(accuracy)


    mean_accuracy = np.mean(fold_accuracies)

    results[n_thresholds] = mean_accuracy

    print( f"Thresholds: {n_thresholds:2d} | " f"CV accuracy: {mean_accuracy:.4f}" )



NameError: name 'StratifiedKFold' is not defined

In [26]:
# ============================================================
# 4. EVALUATE A THRESHOLD CONFIGURATION
# ============================================================
from sklearn.model_selection import StratifiedKFold
def evaluate_configuration(X, y,n_thresholds_per_feature, n_splits=5, epochs=100):

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    accuracies = []

    for train_index, val_index in cv.split(X, y):

        X_train = X[train_index]
        X_val = X[val_index]

        y_train = y[train_index]
        y_val = y[val_index]
mì

        # VERY IMPORTANT:
        # thresholds calculated from training fold only
        thresholds = create_thresholds(
            X_train,
            n_thresholds_per_feature
        )

        # Binarize training and validation data
        X_train_bin = binarize( X_train,thresholds)
        X_val_bin = binarize(X_val,thresholds)

        # Create Tsetlin Machine
        tm = MultiClassTsetlinMachine(number_of_clauses=1000, T=50, s=5.0)

        # Train
        tm.fit(X_train_bin, y_train, epochs=100)

        # Validation prediction
        prediction = tm.predict(X_val_bin)

        accuracy = accuracy_score(y_validation, prediction)

        accuracies.append(accuracy)


    return np.mean(accuracies)
    print(accuracies.append(accuracy))

In [39]:
candidate_thresholds = range(1, 11)

# Start with 1 threshold per feature
best_threshold_counts = [1] * X.shape[1]

for feature_index in range(X.shape[1]):

    best_k = None
    best_score = -1

    for k in candidate_thresholds:

        configuration = best_threshold_counts.copy()

        # Change only this feature
        configuration[feature_index] = k

        score = evaluate_configuration(X_train, X_validation, y_train, y_validation, configuration)

        print(feature_names[feature_index],"k =", k,"accuracy =", score)

        if score > best_score:
            best_score = score
            best_k = k

    best_threshold_counts[feature_index] = best_k

    print("BEST:",feature_names[feature_index],"->", best_k)

TypeError: int() argument must be a string, a bytes-like object or a real number, not 'list'